# Omniscient 2D — does `readable ≠ grabbable` survive FULL observability?

**The thread's question.** Every editability result in this repo has been measured through a **1D
perspective scan**: a fan of rays returning the reflectivity of the *first* surface hit. That
observation is lossy twice over — it **projects** (a 2D world collapses to a 1D signal) and it
**occludes** (only the nearest surface is reported). `orthogonal_edits` (2026-08-05) relocated the
thread's central negative from the models to the **world**, via the `∫gg' = 0` argument: a linear
probe reads an object's *plateau*, moving the object changes only its *edges*, and a plateau is
nearly perpendicular to the spikes at its own edges.

That argument is stated for a 1D scan. This notebook replaces the observation channel with an
**omniscient** one — a top-down orthographic raster in which nothing is projected away and nothing
is hidden — and asks which results survive. If the negative holds when the model was never denied
information, it is not a consequence of an impoverished observation.

**The design is a one-variable swap.** `datasets/12_omniscient2d` is generated with split base seeds
matched to `datasets/4_fixed_refl_inview`, and scene generation is deterministic in the seed, so the
two suites contain **bit-identical** positions, velocities, reflectivities, edit objects and edit
targets. Only the rendering differs. A **sample-matched 1D control** (`1D_H256_30k_s0`, trained on
the same 30 000 scenes) removes the remaining confound of training-set size.

Full config for every run and dataset: **`OMNISCIENT_2D_RUNS.md`** in this directory.
Metric and editor definitions: **`../METRICS_AND_EDITORS.md`** (the canonical registry).
The 2D form of the waterfall spec: **`WATERFALL_SPEC_2D.md`** (+ `frame_grid.py`).

---
## Bootstrap — models, datasets, shared helpers

In [ ]:
# [1] Bootstrap: imports, models, datasets. Every section states its checkpoint/split here.
import sys, os, time, json, warnings, pathlib
# Walk up to the repo root rather than counting "..": this notebook sits four levels
# deep, and a hard-coded relative root silently resolves to the wrong directory.
HERE = pathlib.Path.cwd()
ROOT_P = next(p for p in [HERE, *HERE.parents] if (p / "pim").is_dir() and (p / "datasets").is_dir())
sys.path.insert(0, str(ROOT_P))         # repo root -> import pim, scripts
sys.path.insert(0, str(HERE))           # this dir  -> import frame_grid
import numpy as np, h5py, torch, matplotlib.pyplot as plt
from IPython.display import Markdown, display
warnings.filterwarnings("ignore", category=UserWarning)

from pim.world_models.loader import load_checkpoint, load_dataset
from pim.simulator.config import SimConfig, obs_dim
from pim.simulator.render2d import unflatten, grid_shape, pixel_size
from pim.simulator.renderer import render_scene
from pim.simulator.sim import Scene
from pim.extractors.standard import fit_readability_probes
from pim.extractors.linear import LinearExtractor
from pim.extractors.mlp import MLPExtractor
from pim.extractors.base import StateDefinition
from pim.editors.probe_steering import inject_state, probe_decomposition
from pim.editors.manifold_steering import fit_state_subspace, manifold_steer, project_to_subspace
from pim.eval.controllability import warm_up_to_edit
from scripts.editability_metrics import (build_edit_zones, edit_scorecard, fidelity_ratio,
                                         edit_index, SCORECARD_COLUMNS)
from frame_grid import Arm, frame_animation, frame_grid, frame_trails

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ROOT   = str(ROOT_P)
N_OBJ, K_ROLL, N_EDIT, N_PROBE = 2, 15, 256, 600

# --- the three arms. "channel" is the ONE variable the thread is about. -------------
RUNS = [
    ("2D · omniscient raster · seed 0", f"{ROOT}/runs/omniscient_2d/2D_H256_s0",     "2d", "#0072B2"),
    ("2D · omniscient raster · seed 1", f"{ROOT}/runs/omniscient_2d/2D_H256_s1",     "2d", "#56B4E9"),
    ("1D · perspective scan · 30k (matched)", f"{ROOT}/runs/omniscient_2d/1D_H256_30k_s0", "1d", "#D55E00"),
]
DATA = {"2d": f"{ROOT}/datasets/12_omniscient2d", "1d": f"{ROOT}/datasets/4_fixed_refl_inview"}

BUNDLE = {k: load_dataset(v) for k, v in DATA.items()}
SIM    = {k: b.test.config["dataset"]["sim"] for k, b in BUNDLE.items()}
CFG    = {k: SimConfig(**s) for k, s in SIM.items()}
MODELS = {}
rows = ["| arm | channel | obs dim | hidden | params | best epoch | val loss | train seqs |",
        "|---|---|---|---|---|---|---|---|"]
for name, d, ch, col in RUNS:
    m, info = load_checkpoint(f"{d}/best_model.pt", device=DEVICE)
    MODELS[name] = dict(model=m, info=info, ch=ch, color=col, dir=d)
    tc = info.train_config
    rows.append(f"| {name} | {ch.upper()} | {info.model_config['input_dim']} | {m.hidden_size} | "
                f"{sum(p.numel() for p in m.parameters()):,} | {info.epoch} | {info.val_loss:.5f} | "
                f"{tc.get('n_train_limit') or 30000:,} |")
display(Markdown("**Table 0 — the arms.** All three are the `controls/H256` recipe verbatim "
                 "(H=256, 1 layer, affine decoder, batch 256, lr 1e-3, AdamW wd 1e-4, 400 epochs, "
                 "best-val selection); only the dataset and seed differ.\n\n" + "\n".join(rows)))
print(f"\n2D grid {grid_shape(CFG['2d'])}  pixel {pixel_size(CFG['2d'])[0]:.4f} world units  "
      f"obs dim {obs_dim(CFG['2d'])}   |   1D obs dim {obs_dim(CFG['1d'])}")
print("⚠ val losses are on DIFFERENT observation channels and are NOT comparable across the 1D/2D rows.")

In [ ]:
# [2] Confirm the one-variable claim: the two suites are the SAME WORLDS, rendered differently.
same = {}
for split in ("test", "edits"):
    a = h5py.File(f"{DATA['2d']}/{split}.h5", "r"); b = h5py.File(f"{DATA['1d']}/{split}.h5", "r")
    n = 200
    same[split] = dict(
        seeds     = bool(np.array_equal(a["seeds"][:n],      b["seeds"][:n])),
        positions = bool(np.array_equal(a["positions"][:n],  b["positions"][:n])),
        velocities= bool(np.array_equal(a["velocities"][:n], b["velocities"][:n])),
        refl      = bool(np.array_equal(a["reflectivities"][:n], b["reflectivities"][:n])),
    )
    if split == "edits":
        same[split]["edit_object"] = bool(np.array_equal(a["edit_object"][:n], b["edit_object"][:n]))
        same[split]["edit_value"]  = bool(np.allclose(a["edit_value"][:n],  b["edit_value"][:n]))
    a.close(); b.close()
assert all(all(v.values()) for v in same.values()), same
display(Markdown("**Table 1 — the one-variable check** (first 200 rows of each split). Every world "
                 "quantity is bit-identical between the 1D and omniscient-2D suites; only the "
                 "rendering differs.\n\n"
                 "| split | " + " | ".join(same["edits"]) + " |\n|---|" + "---|"*len(same["edits"]) + "\n"
                 + "\n".join(f"| {s} | " + " | ".join("✅" if same[s].get(k) else ("—" if k not in same[s] else "❌")
                             for k in same["edits"]) + " |" for s in same)))

In [ ]:
# [3] Shared theme + helpers used by every section (light for metrics, dark for generations).
OK = {"blue":"#0072B2","orange":"#D55E00","green":"#009E73","pink":"#CC79A7",
      "yellow":"#E69F00","sky":"#56B4E9","grey":"#999999","black":"#000000"}
def style_ax(ax, grid=True):
    ax.set_facecolor("white")
    for s in ("top","right"): ax.spines[s].set_visible(False)
    for s in ("left","bottom"): ax.spines[s].set_color("#444")
    ax.tick_params(colors="#444", labelsize=8.5)
    if grid: ax.grid(alpha=.25, lw=.6); ax.set_axisbelow(True)
plt.rcParams.update({"figure.facecolor":"white","savefig.facecolor":"white","font.size":9.5})

@torch.no_grad()
def teacher_force(model, obs, bs=256):
    """obs (N,T,R) -> (preds (N,T-1,R), h (N,T-1,H)) in batches."""
    P, H = [], []
    for i in range(0, len(obs), bs):
        o = torch.from_numpy(obs[i:i+bs]).float().to(DEVICE)
        p, h = model.observe_sequence(o)
        P.append(p.cpu().numpy()); H.append(h.cpu().numpy())
    return np.concatenate(P), np.concatenate(H)

def rmse(a, b): return float(np.sqrt(((a - b) ** 2).mean()))

---
## Definitions & metrics (the invariant spine — read once)

Names, formulas and units are copied from **`../METRICS_AND_EDITORS.md`**, the canonical registry.
Results that *move* are **not** here — each section carries its own dated
`Current results` block.

| term / metric | definition & formula | units | better |
|---|---|---|---|
| **observation channel** | the map world-state → observation. **1D scan**: 128 rays from the origin, each returning the *first-hit* reflectivity (projects **and** occludes). **Omniscient 2D**: a 48×64 orthographic raster over `x∈[-6,6] × y∈[3,12]`, flattened row-major to 3072 (no projection, no occlusion, no perspective). | — | — |
| `(pos, vel)` | per-object position & velocity; 2 objects → an **8-dim** minimal sufficient statistic (the sim is exactly constant-velocity) | sim units | — |
| **noise floor** | `RMSE(obs, clean_obs)` — the observation noise, as a *reference scale*, never a bound. A recurrent model legitimately scores **below** it by denoising. | obs | — |
| **next-step RMSE** | teacher-forced `RMSE(pred, clean_obs[1:])` — **always vs clean**, never vs noisy | obs | ↓ |
| PCA hull @p% | # PCA components of the visited-`h` bank reaching ≥ p% variance (a linear *upper* bound on intrinsic dim) | dims | — |
| intrinsic dim (TwoNN) | `d = 1/mean(log(r₂/r₁))`, r₁,r₂ = 1st/2nd-NN distances (Facco 2017) | dims | — |
| intrinsic dim (MLE) | Levina–Bickel over k=20 NN, ×(k−2)/(k−1) | dims | — |
| **position / velocity R²** | `1 − ‖Y − probe(h)‖²/‖Y − Ȳ‖²`, fit by `pim.extractors.fit_readability_probes`: linear lstsq **and** a 2×256 ReLU MLP (300 epochs), both on the same 80 % of **sequences**, both scored on the held-out 20 % against the **train** mean | — | ↑ |
| **fiber residual** | `‖h − g(pos,vel)‖ / ‖h‖`, `g` linear or MLP. 0 = `h` is a pure function of the physical statistic (fully canonical) | frac of ‖h‖ | ↓ |
| **edit zones** | from the two ground-truth worlds at `ef`. **target** = pixels the edited object occupies in `gt_edited`; **ghost** = pixels it occupied pre-edit and must vacate; **collateral** = the *other* object's pixels; **differing** = where the two worlds differ (`>1e-3`) — the support of the index | — | — |
| **Target / Ghost / Collateral / Edit-frame RMSE** | `RMSE(edited₀, gt_edited)` restricted to that zone (Edit-frame = all pixels), at rollout **step 0**, which decodes frame `ef` | obs | ↓ |
| **GT-traj RMSE** | `mean_s RMSE(edited_s, clean_obs[ef+s])` over the K=15 rollout — did the edit *hold*? | obs | ↓ |
| **fidelity ratio** | `GT-traj RMSE(editor) / GT-traj RMSE(unsteered)`. **> 1 = the edit left the model FURTHER from the true post-edit world than doing nothing.** Always report beside any success claim. | ratio | ↓ |
| **Edit Index** | `(d_uned − d_edit)/(d_uned + d_edit)`, `d_· = RMSE(edited₀, gt_·)` over **differing** pixels, per sample then averaged. **+1** = output *is* the edited world · **0** = equidistant (ambiguous **or garbage**) · **−1** = *is* the unedited world | −1…+1 | ↑ |
| **Edit Index by step** | the same at every rollout step, against the counterfactual world **rolled forward** | −1…+1 | ↑ |

**Read the index against that model's own unsteered row.** A perfect predictor scores exactly −1
unsteered; a real one falls short by its own blur, because `d_unedited` is its one-step prediction
error rather than 0. The unsteered row therefore appears in every table.

> ### ⚠ Whole-frame errors are NOT comparable across observation channels
> An object covers ~13 % of a 1D scan at mid-depth but **0.73 %** of the omniscient grid — an ~18×
> dilution — so any average over *all* rays/pixels (**next-step RMSE**, **Edit-frame RMSE**,
> **GT-traj RMSE**, and hence the **fidelity ratio**'s numerator and denominator individually) is
> dominated by background to a wildly different degree in each channel.
> **Cross-channel reading is restricted to:** the **Edit Index** (defined only on differing pixels
> and bounded), and each metric's **ratio to its own channel's reference** (a model's index against
> its own unsteered row; an editor's fidelity ratio, where the channel-dependent scale cancels).
> Within one channel every metric is comparable as usual. Figures below mark cross-channel
> comparisons explicitly.

**Editors** (canonical names, `../METRICS_AND_EDITORS.md`). *Standard* = training-free writes to `h`;
*oracle* = given ground-truth access. **Pseudoinverse Injection** `Δ = A⁺(target − (Ah+b))`;
**Global PCA Projection (PI)** = alternating inject ↔ project onto the 90 %-variance subspace (POCS,
50 rounds); **MLP Grad Steering** = Adam on `h` through a *frozen* 1×128 `MLPExtractor` on (pos,vel);
**First Obs. TF** = teacher-force one frame, the real **noisy** `edits.obs[ef]` (**leads every other
column by one frame**); **Counterfactual Overwriting** = teacher-force a fabricated history in which
the object always travelled toward the target; **Freeze-time Interp. TF @8** = freeze the world and
teacher-force 8 **externally rendered** interpolation frames; **Decoder Grad Steering k=1 / k=15** =
Adam on `h` so the decoder renders the GT edit frame / so the whole 15-step rollout matches the GT
sequence.

---
## §0 — The two observation channels, side by side

**What this section establishes.** What the model actually sees in each channel, the reference
scales that make later numbers readable (noise floor, object occupancy), and the predictive-quality
gate: each model must be a competent next-step predictor *in its own channel* before anything about
its latent means much.

In [ ]:
# [4] §0 — channel reference scales + next-step predictive quality gate (each vs its OWN clean obs).
Q, TF = {}, {}
for name, M in MODELS.items():
    ch = M["ch"]; t = BUNDLE[ch].test
    obs, clean = t.obs[:N_PROBE], t.clean_obs[:N_PROBE]
    pred, h = teacher_force(M["model"], obs)
    TF[name] = dict(h=h, pos=t.positions[:N_PROBE, :-1], vel=None)
    Q[name] = dict(next_step_clean=rmse(pred, clean[:, 1:]),
                   next_step_noisy=rmse(pred, obs[:, 1:]),
                   noise_floor=rmse(obs, clean),
                   occupancy=float((t.obs_id[:N_PROBE] >= 0).mean()))
rows = ["| arm | channel | next-step RMSE **vs clean** ↓ | noise floor (reference) | ratio to noise floor | occupied fraction of the frame |",
        "|---|---|---|---|---|---|"]
for name, M in MODELS.items():
    q = Q[name]
    rows.append(f"| {name} | {M['ch'].upper()} | **{q['next_step_clean']:.4f}** | {q['noise_floor']:.4f} | "
                f"{q['next_step_clean']/q['noise_floor']:.2f}× | {q['occupancy']*100:.2f} % |")
display(Markdown("**Table 2 — predictive-quality gate.** Every arm predicts well below its own "
                 "channel's noise floor, i.e. it denoises rather than echoes. **The RMSE column is "
                 "NOT comparable between the 1D and 2D rows** (see the dilution warning above) — the "
                 "ratio-to-floor column is the like-for-like read.\n\n" + "\n".join(rows)))
print("occupancy: fraction of rays/pixels covered by an object — the 18x dilution, measured.")

In [ ]:
# [5] Fig 0 — what the model sees in each channel (same scene, same frame, both renderings).
t2, t1 = BUNDLE["2d"].test, BUNDLE["1d"].test
si, fr = 3, 20
fig = plt.figure(figsize=(13.2, 3.5)); fig.patch.set_facecolor("#0a0a14")
gs = fig.add_gridspec(1, 4, width_ratios=[1.25, 1.25, 1.6, 1.6], wspace=.28)
for k, (arr, ttl) in enumerate([(t2.clean_obs, "(a) omniscient 2D — clean"),
                                (t2.obs,       "(b) omniscient 2D — noisy (what the model sees)")]):
    ax = fig.add_subplot(gs[0, k])
    ax.imshow(unflatten(arr[si, fr], CFG["2d"]), cmap="gray", vmin=0, vmax=1, origin="lower",
              extent=(-6, 6, 3, 12), aspect="equal", interpolation="nearest")
    ax.set_title(ttl, color="w", fontsize=9); ax.tick_params(colors="#a3adc2", labelsize=7)
    ax.set_xlabel("x (world)", color="#a3adc2", fontsize=8)
    if k == 0: ax.set_ylabel("y — depth (world)", color="#a3adc2", fontsize=8)
for k, (arr, ttl) in enumerate([(t1.clean_obs, "(c) 1D perspective scan — clean"),
                                (t1.obs,       "(d) 1D scan — noisy")]):
    ax = fig.add_subplot(gs[0, 2 + k]); ax.set_facecolor("#0a0a14")
    ax.plot(arr[si, fr], color="#56B4E9", lw=1.1)
    ax.set_ylim(-.02, 1.02); ax.set_title(ttl, color="w", fontsize=9)
    ax.tick_params(colors="#a3adc2", labelsize=7); ax.set_xlabel("ray index", color="#a3adc2", fontsize=8)
    for s in ("top","right"): ax.spines[s].set_visible(False)
    for s in ("left","bottom"): ax.spines[s].set_color("#5c677f")
    if k == 0: ax.set_ylabel("intensity", color="#a3adc2", fontsize=8)
fig.suptitle("Fig 0 — the same world state in both observation channels (test sample "
             f"{si}, frame {fr})", color="w", fontsize=11, y=1.04)
plt.show()
print(f"GT positions at this frame: {t2.positions[si, fr]}")
print("Both objects are fully present in (a)/(b). In (c)/(d) the scan reports only first-hit "
      "reflectivity along each ray — depth and any occluded surface are gone.")

---
## §1 — Geometry: how many degrees of freedom does the visited-state manifold have?

**What this section measures.** Three reads on the bank of visited hidden states `h`:
the **linear-hull dimension** (PCA components for 90 % of variance — an *upper* bound, since a
curved low-dim surface needs more linear dims than its intrinsic dim), and two **model-free
intrinsic-dimension** estimators (TwoNN, MLE). The physical statistic is 8-dimensional, so the gap
between 8 and these numbers is what the state carries beyond `(pos, vel)`.

Both estimators are computed **for every arm on its own bank**, with the same estimator and the same
bank size, so the numbers are comparable across arms.

In [ ]:
# [6] §1 — PCA hull + model-free intrinsic dimension (TwoNN, MLE) per arm, on matched bank sizes.
BANK_N = 20000
def scree(bank):
    X = bank - bank.mean(0); C = np.cov(X, rowvar=False)
    ev = np.sort(np.linalg.eigvalsh(C))[::-1]; ev = np.clip(ev, 0, None)
    return np.cumsum(ev) / ev.sum()
def _knn_dists(X, k):
    """Sorted distances to the k nearest OTHER points, via torch (scipy is not in .pim)."""
    T = torch.from_numpy(np.ascontiguousarray(X)).float().to(DEVICE)
    d = torch.cdist(T, T)
    d.fill_diagonal_(float("inf"))
    return torch.topk(d, k, dim=1, largest=False).values.cpu().numpy().astype(np.float64)
def twonn(X, frac=0.9):
    """Facco et al. 2017: d = slope of -log(1-F) on log(r2/r1)."""
    d = _knn_dists(X, 2); r1, r2 = d[:, 0], d[:, 1]
    ok = (r1 > 1e-12) & (r2 > r1); mu = np.sort(r2[ok] / r1[ok])
    m = int(frac * len(mu)); F = np.arange(1, m + 1) / len(mu)
    return float(np.linalg.lstsq(np.log(mu[:m])[:, None], -np.log(1 - F), rcond=None)[0][0])
def mle_id(X, k=20):
    """Levina-Bickel MLE over k nearest neighbours, with the (k-2)/(k-1) bias correction."""
    d = np.clip(_knn_dists(X, k), 1e-12, None)
    inv = (np.log(d[:, -1:] / d[:, :-1])).mean(1)
    return float((1.0 / inv).mean() * (k - 2) / (k - 1))

GEO = {}
rng = np.random.default_rng(0)
for name, M in MODELS.items():
    h = TF[name]["h"]; bank = h.reshape(-1, h.shape[-1])
    idx = rng.choice(len(bank), min(BANK_N, len(bank)), replace=False)
    B = bank[idx].astype(np.float64)
    cs = scree(B)
    GEO[name] = dict(cum=cs, pca70=int(np.searchsorted(cs, .70) + 1),
                     pca90=int(np.searchsorted(cs, .90) + 1), pca95=int(np.searchsorted(cs, .95) + 1),
                     twonn=twonn(B[:8000]), mle=mle_id(B[:8000]))
rows = ["| arm | PCA hull @70% | @90% | @95% | intrinsic dim (TwoNN) | intrinsic dim (MLE) |","|---|---|---|---|---|---|"]
for name in MODELS:
    g = GEO[name]
    rows.append(f"| {name} | {g['pca70']} | **{g['pca90']}** | {g['pca95']} | {g['twonn']:.1f} | {g['mle']:.1f} |")
display(Markdown(f"**Table 3 — geometry of the visited-state bank.** H=256, bank = {BANK_N:,} states "
                 "subsampled from teacher-forced test sequences (same size for every arm). The "
                 "physical statistic is **8**-dimensional.\n\n" + "\n".join(rows)))

In [ ]:
# [7] Fig 1 — geometry, built to hold N arms: (a) PCA scree, (b) intrinsic-dim estimators.
fig, axes = plt.subplots(1, 2, figsize=(12.4, 4.0))
ax = axes[0]
for name, M in MODELS.items():
    ax.plot(np.arange(1, 61), GEO[name]["cum"][:60], color=M["color"], lw=1.8, label=name)
ax.axhline(.90, color=OK["grey"], ls="--", lw=1)
ax.text(60, .905, "90% of variance", ha="right", fontsize=8, color="#444")
ax.axvline(8, color=OK["green"], ls=":", lw=1.4)
ax.text(8.6, .32, "8 = physical\nstatistic", fontsize=8, color=OK["green"])
ax.set_xlabel("PCA component"); ax.set_ylabel("cumulative variance explained")
ax.set_title("(a) linear hull of the visited-state bank", fontsize=10)
ax.legend(fontsize=7.5, frameon=False, loc="lower right"); style_ax(ax)

ax = axes[1]
ests = ["pca90", "twonn", "mle"]; lbl = ["PCA hull @90%\n(linear upper bound)", "intrinsic dim\n(TwoNN)", "intrinsic dim\n(MLE)"]
y = np.arange(len(ests)); hgt = .8 / len(MODELS)
for j, (name, M) in enumerate(MODELS.items()):
    ax.barh(y + j * hgt - .4 + hgt / 2, [GEO[name][e] for e in ests], height=hgt,
            color=M["color"], label=name)
ax.axvline(8, color=OK["green"], ls=":", lw=1.4); ax.text(8.5, -.45, "8", color=OK["green"], fontsize=9)
ax.set_yticks(y); ax.set_yticklabels(lbl, fontsize=8.5)
ax.set_xlabel("dimensions"); ax.set_title("(b) dimensionality estimates (horizontal bars: long labels)", fontsize=10)
ax.legend(fontsize=7.5, frameon=False, loc="lower right"); style_ax(ax)
fig.suptitle("Fig 1 — geometry of the visited hidden-state manifold, per arm", fontsize=11.5, y=1.02)
plt.tight_layout(); plt.show()

---
## §2 — Recoverability: can `(pos, vel)` be read out of a single hidden state?

**What this section measures.** Given one hidden state `h_t`, how well does a probe recover the
physical statistic? Both probes come from `pim.extractors.fit_readability_probes` — linear lstsq
and a 2×256 ReLU MLP, fit on the same 80 % of **sequences** and scored on the same held-out 20 %
against the train mean. An in-sample R² is not a readability claim, and a by-row split would leak
near-duplicate neighbouring frames across the boundary.

R² is dimensionless and defined against the same physical targets in both channels, so **§2 is one
of the places a 1D↔2D comparison is legitimate**.

In [ ]:
# [8] §2 — position and velocity readout per arm (standard probes; held-out by sequence).
REC = {}
for name, M in MODELS.items():
    ch = M["ch"]; t = BUNDLE[ch].test
    h = TF[name]["h"]
    v = h5py.File(t.h5_path, "r")["velocities"][:N_PROBE, :, :N_OBJ, :].astype(np.float32)
    pos = t.positions[:N_PROBE, :-1].reshape(N_PROBE, -1, N_OBJ * 2)
    vel = v[:, :-1].reshape(N_PROBE, -1, N_OBJ * 2)
    late = np.zeros(pos.shape[:2], bool); late[:, 15:] = True    # late-t = frames t >= 15 (filter converged)
    REC[name] = dict(
        pos      = fit_readability_probes(h, pos, device=DEVICE),
        vel      = fit_readability_probes(h, vel, device=DEVICE),
        vel_late = fit_readability_probes(h, vel, mask=late, device=DEVICE),
    )
rows = ["| arm | position R² (linear) | position R² (MLP) | velocity R² (linear) | velocity R² (MLP) | velocity R² MLP, late-t (t≥15) |",
        "|---|---|---|---|---|---|"]
for name in MODELS:
    r = REC[name]
    rows.append(f"| {name} | **{r['pos']['linear_r2']:.3f}** | **{r['pos']['mlp_r2']:.3f}** | "
                f"{r['vel']['linear_r2']:.3f} | {r['vel']['mlp_r2']:.3f} | {r['vel_late']['mlp_r2']:.3f} |")
display(Markdown(f"**Table 4 — recoverability of the physical statistic from a single `h`.** "
                 f"{REC[list(MODELS)[0]]['pos']['spec']}. N={N_PROBE} sequences. R² is dimensionless "
                 "and scored against the same physical targets in both channels, so these rows **are** "
                 "cross-channel comparable.\n\n" + "\n".join(rows)))

In [ ]:
# [9] Fig 2 — recoverability, one bar per arm per probe (built to hold N arms).
fig, axes = plt.subplots(1, 2, figsize=(12.4, 4.0))
for ax, key, ttl in [(axes[0], "pos", "(a) position readout from a single h"),
                     (axes[1], "vel", "(b) velocity readout from a single h")]:
    cats = ["linear (lstsq)", "MLP (2×256)"]; y = np.arange(len(cats)); hgt = .8 / len(MODELS)
    for j, (name, M) in enumerate(MODELS.items()):
        vals = [REC[name][key]["linear_r2"], REC[name][key]["mlp_r2"]]
        ax.barh(y + j * hgt - .4 + hgt / 2, vals, height=hgt, color=M["color"], label=name)
    ax.set_yticks(y); ax.set_yticklabels(cats, fontsize=9)
    ax.set_xlabel("held-out R²  (↑ better)"); ax.set_xlim(0, 1.0)
    ax.set_title(ttl, fontsize=10); style_ax(ax)
axes[0].legend(fontsize=7.5, frameon=False, loc="lower right")
fig.suptitle("Fig 2 — recoverability of (pos, vel) from a single hidden state, by observation channel",
             fontsize=11.5, y=1.02)
plt.tight_layout(); plt.show()

---
## §3 — Canonicality: is the hidden state a *function* of `(pos, vel)`?

**What this section measures.** `h` is **canonical** with respect to the physical statistic if it is
a function of it — every state with the same `(pos, vel)` maps to the same `h`. We fit the best map
`g(pos, vel) → h` (linear, and an MLP) and report the residual fraction `‖h − g(pos,vel)‖/‖h‖`.
A residual of 0 means fully canonical; a large linear→MLP drop means the embedding is curved rather
than non-canonical. Whatever the residual leaves unexplained is the part of `h` that a probe over
physical state cannot address — and `delta_h_analysis` found that a *successful* edit lives almost
entirely there.

The residual is a fraction of `‖h‖`, so it is **cross-channel comparable**.

In [ ]:
# [10] §3 — fiber residual: fit g(pos,vel) -> h, linear and MLP, per arm.
FIB = {}
for name, M in MODELS.items():
    ch = M["ch"]; t = BUNDLE[ch].test
    h = TF[name]["h"]
    v = h5py.File(t.h5_path, "r")["velocities"][:N_PROBE, :, :N_OBJ, :].astype(np.float32)
    pv = np.concatenate([t.positions[:N_PROBE, :-1].reshape(N_PROBE, -1, N_OBJ*2),
                         v[:, :-1].reshape(N_PROBE, -1, N_OBJ*2)], -1)      # (N, T-1, 8)
    g = fit_readability_probes(pv, h, device=DEVICE)      # note: INPUT is (pos,vel), TARGET is h
    n_tr = g["n_train_seq"]
    Xte = pv[n_tr:].reshape(-1, 8); Yte = h[n_tr:].reshape(-1, h.shape[-1])
    lin = Xte @ g["A"].T + g["b"]
    with torch.no_grad():
        mlp = g["mlp"](torch.tensor(Xte, dtype=torch.float32, device=DEVICE)).cpu().numpy()
    nrm = np.linalg.norm(Yte, axis=1).mean()
    FIB[name] = dict(lin=float(np.linalg.norm(Yte - lin, axis=1).mean() / nrm),
                     mlp=float(np.linalg.norm(Yte - mlp, axis=1).mean() / nrm),
                     r2_lin=g["linear_r2"], r2_mlp=g["mlp_r2"])
rows = ["| arm | fiber residual (linear g) | fiber residual (MLP g) | R² on h (linear) | R² on h (MLP) |","|---|---|---|---|---|"]
for name in MODELS:
    f = FIB[name]
    rows.append(f"| {name} | {f['lin']:.3f} | **{f['mlp']:.3f}** | {f['r2_lin']:.3f} | {f['r2_mlp']:.3f} |")
display(Markdown("**Table 5 — fiber-collapse residual**, `‖h − g(pos,vel)‖/‖h‖`, held-out by sequence. "
                 "0 = `h` is a pure function of the 8-dim physical statistic. A fraction of ‖h‖, hence "
                 "cross-channel comparable.\n\n" + "\n".join(rows)))

In [ ]:
# [11] Fig 3 — fiber residual per arm, linear vs MLP g.
fig, ax = plt.subplots(figsize=(8.4, 3.6))
cats = ["linear  g(pos,vel)→h", "MLP  g(pos,vel)→h"]; y = np.arange(2); hgt = .8 / len(MODELS)
for j, (name, M) in enumerate(MODELS.items()):
    ax.barh(y + j*hgt - .4 + hgt/2, [FIB[name]["lin"], FIB[name]["mlp"]], height=hgt,
            color=M["color"], label=name)
ax.set_yticks(y); ax.set_yticklabels(cats, fontsize=9)
ax.set_xlabel("residual as a fraction of ‖h‖   (↓ = more canonical; 0 = h is a function of (pos,vel))")
ax.set_title("Fig 3 — how much of the hidden state is NOT explained by the physical statistic", fontsize=10.5)
ax.legend(fontsize=7.5, frameon=False, loc="lower right"); style_ax(ax)
plt.tight_layout(); plt.show()

---
## §4 — Editing head-to-head (the core comparison)

**Protocol**, identical for every arm and every editor. Teacher-force each edits-split sequence to
`edit_frame = 20` on the **pre-edit** observations, apply the editor to the hidden state `h`, then
roll the model out freely for **K = 15** steps. Rollout **step 0 decodes frame `ef`**, so
`ROLL[:, 0:K]` is scored against `clean_obs[ef:ef+K]` — no slicing, no dropped step. The intended
outcome is the true post-edit world; every metric is scored against the simulator's **clean** render.

Each model is evaluated on **its own channel's** edits split — which, per Table 1, contains the
identical scenes and the identical teleports.

In [ ]:
# [12] §4 — shared edit-set setup per channel: zones, GT rollout, warm-up states, probes, banks.
E, ZONES, GTROLL, WARM, PROBES = {}, {}, {}, {}, {}
for ch in ("2d", "1d"):
    e = BUNDLE[ch].edits; ef = e.edit_frame
    v = h5py.File(e.h5_path, "r")["velocities"][:N_EDIT, :, :N_OBJ, :].astype(np.float32)
    pre, tgt, pvel = e.positions[:N_EDIT, ef-1], e.positions[:N_EDIT, ef], v[:, ef-1]
    oe = e.edit_object[:N_EDIT].astype(int); idx = np.arange(N_EDIT)
    ZONES[ch] = build_edit_zones(pre_pos=pre, tgt_pos=tgt, pre_vel=pvel, edit_object=oe, sim=SIM[ch],
                                 traj_pos=e.positions[:N_EDIT, ef:ef+K_ROLL],
                                 gt_edited_traj=e.clean_obs[:N_EDIT, ef:ef+K_ROLL])
    GTROLL[ch] = e.clean_obs[:N_EDIT, ef:ef+K_ROLL].astype(np.float32)
    E[ch] = dict(e=e, ef=ef, pre=pre, tgt=tgt, pvel=pvel, oe=oe, idx=idx, vel_all=v,
                 target_xy=tgt[idx, oe], ghost_xy=pre[idx, oe] + pvel[idx, oe]*float(SIM[ch]["dt"]))
    z = ZONES[ch]
    print(f"{ch.upper()}: ef={ef}  zones/sample — target {z.target.sum(1).mean():5.1f}  "
          f"ghost {z.ghost.sum(1).mean():5.1f}  collateral {z.collateral.sum(1).mean():5.1f}  "
          f"differing {z.differing.sum(1).mean():5.1f}   (of R={GTROLL[ch].shape[-1]})")

sdef_p  = StateDefinition(name="pos",   state_shape=(N_OBJ, 2), extract_fn=lambda x: x)
sdef_pv = StateDefinition(name="posvel", state_shape=(N_OBJ*4,), extract_fn=lambda x: x)
for name, M in MODELS.items():
    ch = M["ch"]; m = M["model"]; d = E[ch]
    wu = warm_up_to_edit(m, d["e"].obs[:N_EDIT], d["ef"], device=DEVICE, desc=f"warm-up {name}")
    H0 = torch.from_numpy(wu.h_at_edit).float().to(DEVICE)
    h = TF[name]["h"]; t = BUNDLE[ch].test
    lin = LinearExtractor(m.hidden_size, sdef_p); lin.fit(h, t.positions[:N_PROBE, :-1], device=DEVICE)
    A, b_, A_pinv = probe_decomposition(lin.to(DEVICE).eval())
    v = h5py.File(t.h5_path, "r")["velocities"][:N_PROBE, :, :N_OBJ, :].astype(np.float32)
    pv = np.concatenate([t.positions[:N_PROBE, :-1].reshape(N_PROBE,-1,N_OBJ*2),
                         v[:, :-1].reshape(N_PROBE,-1,N_OBJ*2)], -1)
    mlp_pv = MLPExtractor(m.hidden_size, sdef_pv)      # frozen 1x128 STEERING probe (NOT the reporting probe)
    mlp_pv.fit(h, pv, device=DEVICE); mlp_pv = mlp_pv.to(DEVICE).eval()
    bank = torch.from_numpy(h.reshape(-1, h.shape[-1])[::7].copy()).float().to(DEVICE)
    WARM[name] = dict(H0=H0, wu=wu)
    PROBES[name] = dict(lin=lin, A=A, b=b_, A_pinv=A_pinv, mlp_pv=mlp_pv, bank=bank,
                        sub=fit_state_subspace(bank, var_threshold=0.90))
print("\nMLP Grad Steering uses a frozen 1x128 MLPExtractor — a DIFFERENT object from the 2x256 "
      "reporting probe of §2. Never quote one as the other (METRICS_AND_EDITORS.md §2).")

In [ ]:
# [13] §4 — externally-rendered observation sequences for the oracle editors (per channel).
#      render_scene dispatches on cfg, so the SAME code renders 1D scans and omniscient 2D rasters.
N_FT = 8
CF_OBS, FT_OBS = {}, {}
for ch in ("2d", "1d"):
    d, s = E[ch], SIM[ch]; ef = d["ef"]; R = GTROLL[ch].shape[-1]
    REFL = np.array([s["refl_min"], s["refl_max"]], np.float32)
    RAD  = np.full(N_OBJ, s["radius"], np.float32); COLc = np.ones((N_OBJ, 3), np.float32)
    def _cfg(nf, noise, s=s):
        return SimConfig(seed=0, y_near=s["y_near"], y_far=s["y_far"], x_near=s["x_near"], x_far=s["x_far"],
                         n_objects=N_OBJ, radius=s["radius"], n_frames=nf, dt=s["dt"], obs_res=s["obs_res"],
                         refl_min=s["refl_min"], refl_max=s["refl_max"], fixed_reflectivities=True,
                         obs_noise_std=noise, boundary="open", always_in_frustum=False,
                         omni2d=s.get("omni2d", False), omni2d_h=s.get("omni2d_h", 48),
                         omni2d_w=s.get("omni2d_w", 64))
    def render_traj(pos_seq, noise=0.0):
        return render_scene(Scene(positions=pos_seq, velocities=np.zeros_like(pos_seq), radii=RAD,
                                  colors=COLc, reflectivities=REFL,
                                  config=_cfg(len(pos_seq), noise)))[2].astype(np.float32)
    cf = np.zeros((N_EDIT, ef, R), np.float32); ft = np.zeros((N_EDIT, N_FT, R), np.float32)
    ti = np.arange(ef)
    for i in range(N_EDIT):
        o, other = d["oe"][i], 1 - d["oe"][i]
        vv = d["vel_all"][i, ef, o]
        c = np.zeros((ef, N_OBJ, 2), np.float32)
        c[:, o]     = d["tgt"][i, o][None, :] - vv[None, :] * (ef - ti)[:, None] * float(s["dt"])
        c[:, other] = d["e"].positions[i, :ef, other]
        cf[i] = render_traj(c)
        f_ = np.zeros((N_FT, N_OBJ, 2), np.float32)
        for j in range(N_FT):
            f_[j, o] = d["pre"][i, o] + ((j+1)/N_FT) * (d["tgt"][i, o] - d["pre"][i, o])
            f_[j, other] = d["tgt"][i, other]
        ft[i] = render_traj(f_, float(s["obs_noise_std"]))
    CF_OBS[ch], FT_OBS[ch] = cf, ft
    print(f"{ch.upper()}: counterfactual history {cf.shape}, freeze-time frames {ft.shape}")

In [ ]:
# [14] §4 — run every editor on every arm. Same line-up, same metrics, same units, per channel.
@torch.no_grad()
def roll_out(model, h_flat, k=K_ROLL):
    st = model.state_from_flat(h_flat)
    out = [model.decode(st)]                      # step 0 = decode WITHOUT advancing -> frame ef
    for _ in range(k - 1):
        p, st = model.predict_step(st); out.append(p)
    return torch.stack(out, 1).cpu().numpy()

@torch.no_grad()
def continue_from(model, h_flat, frames):
    st = model.state_from_flat(h_flat); o = torch.from_numpy(frames).float().to(DEVICE)
    for t in range(frames.shape[1]):
        _, st = model.step(o[:, t], st)
    return model.flat_state(st)

@torch.no_grad()
def warm_from(model, frames):
    st = None; o = torch.from_numpy(frames).float().to(DEVICE)
    for t in range(frames.shape[1]):
        _, st = model.step(o[:, t], st)
    return model.flat_state(st)

def mlp_grad_steer(h0, target, probe, n_steps=200, lr=0.01):
    """Batched Adam on h through a frozen probe. Adam is invariant to a global gradient
    rescaling, so batching is equivalent to the per-sample loop used in editor_gallery."""
    h = h0.clone().detach().requires_grad_(True)
    opt = torch.optim.Adam([h], lr=lr)
    for _ in range(n_steps):
        opt.zero_grad()
        ((probe(h).reshape(len(h), -1) - target) ** 2).mean().backward()
        opt.step()
    return h.detach()

def decoder_grad(model, h0, gt_ef, gt_seq, k, n_iter=400, lr=0.05):
    h = h0.clone().detach().requires_grad_(True)
    opt = torch.optim.Adam([h], lr=lr)
    with torch.backends.cudnn.flags(enabled=False):
        for _ in range(n_iter):
            st = model.state_from_flat(h)
            if k == 1:
                loss = ((model.decode(st) - gt_ef) ** 2).mean()
            else:
                outs = [model.decode(st)]
                for _s in range(k - 1):
                    p, st = model.predict_step(st); outs.append(p)
                loss = ((torch.stack(outs, 1) - gt_seq) ** 2).mean()
            opt.zero_grad(); loss.backward(); opt.step()
    return h.detach()

STD_ED = ["Pseudoinverse Injection", "Global PCA Projection (PI)", "MLP Grad Steering"]
ORC_ED = ["First Obs. TF", "Freeze-time Interp. TF @8", "Counterfactual Overwriting",
          "Decoder Grad Steering k=1", "Decoder Grad Steering k=15"]
CARDS, ROLLS = {}, {}
t0 = time.perf_counter()
for name, M in MODELS.items():
    ch, m, d = M["ch"], M["model"], E[M["ch"]]
    P, H0 = PROBES[name], WARM[name]["H0"]
    z, gtr, ef = ZONES[ch], GTROLL[ch], E[ch]["ef"]
    tgt   = torch.from_numpy(d["tgt"].reshape(N_EDIT, -1).astype(np.float32)).to(DEVICE)
    tgtpv = torch.from_numpy(np.concatenate([d["tgt"].reshape(N_EDIT, -1),
              d["vel_all"][:, ef].reshape(N_EDIT, -1)], -1).astype(np.float32)).to(DEVICE)
    gt_ef  = torch.from_numpy(d["e"].clean_obs[:N_EDIT, ef]).float().to(DEVICE)
    gt_seq = torch.from_numpy(gtr).float().to(DEVICE)

    def reg(ed, h):
        rl = roll_out(m, h); ROLLS[(name, ed)] = rl
        c = edit_scorecard(rl, z, gtr)
        c["fidelity_ratio"] = 1.0 if ed == "Unsteered" else fidelity_ratio(c, CARDS[(name, "Unsteered")])
        CARDS[(name, ed)] = c; return c

    reg("Unsteered", H0)
    reg("Pseudoinverse Injection", inject_state(H0, tgt, P["A"], P["A_pinv"], P["b"]))
    reg("Global PCA Projection (PI)",
        manifold_steer(H0, tgt, lambda h, t: inject_state(h, t, P["A"], P["A_pinv"], P["b"]),
                       P["sub"], n_iters=50))
    reg("MLP Grad Steering", mlp_grad_steer(H0, tgtpv, P["mlp_pv"]))
    reg("First Obs. TF", continue_from(m, H0, d["e"].obs[:N_EDIT, ef:ef+1].astype(np.float32)))
    reg("Freeze-time Interp. TF @8", continue_from(m, H0, FT_OBS[ch]))
    reg("Counterfactual Overwriting", warm_from(m, CF_OBS[ch]))
    reg("Decoder Grad Steering k=1",  decoder_grad(m, H0, gt_ef, gt_seq, 1))
    reg("Decoder Grad Steering k=15", decoder_grad(m, H0, gt_ef, gt_seq, K_ROLL))
    print(f"{name}: unsteered {CARDS[(name,'Unsteered')]['edit_index']:+.2f}  "
          f"[{time.perf_counter()-t0:.0f}s]")
print(f"\nall editors, all arms: {time.perf_counter()-t0:.0f}s")

In [ ]:
# [15] §4 — the scorecard. Same metric set, same units, every arm; per METRICS_AND_EDITORS.md §4.
for name, M in MODELS.items():
    hdr = ["| family | editor | Edit Index ↑ | at step 14 | Target RMSE ↓ | Ghost RMSE ↓ | "
           "Collateral RMSE ↓ | GT-traj RMSE ↓ | fidelity ↓ |", "|---|---|---|---|---|---|---|---|---|"]
    for fam, eds in [("reference", ["Unsteered"]), ("standard", STD_ED), ("oracle", ORC_ED)]:
        for ed in eds:
            c = CARDS[(name, ed)]
            lead = " *(leads by 1 frame)*" if ed == "First Obs. TF" else ""
            hdr.append(f"| {fam} | {ed}{lead} | **{c['edit_index']:+.2f}** | {c['edit_index_by_step'][-1]:+.2f} | "
                       f"{c['target_rmse']:.3f} | {c['ghost_rmse']:.3f} | {c['collateral_rmse']:.3f} | "
                       f"{c['gt_traj_rmse']:.3f} | {c['fidelity_ratio']:.2f} |")
    display(Markdown(f"**Table 6 — editor scorecard · {name}.** N={N_EDIT} held-out edits, K={K_ROLL}. "
                     "Read the index against this arm's own **Unsteered** row. "
                     "**fidelity > 1 = the edit left the model further from the true post-edit world "
                     "than doing nothing.** RMSE columns are within-channel only.\n\n" + "\n".join(hdr)))

In [ ]:
# [16] Fig 4 — editor head-to-head across arms. (a) Edit Index is the CROSS-CHANNEL panel;
#      (b) each editor's gain over its own arm's unsteered row; (c) persistence over the rollout.
eds = ["Unsteered"] + STD_ED + ORC_ED
fig, axes = plt.subplots(1, 3, figsize=(16.5, 5.0))
y = np.arange(len(eds)); hgt = .8 / len(MODELS)
ax = axes[0]
for j, (name, M) in enumerate(MODELS.items()):
    ax.barh(y + j*hgt - .4 + hgt/2, [CARDS[(name, e)]["edit_index"] for e in eds],
            height=hgt, color=M["color"], label=name)
ax.axvline(0, color="#444", lw=1)
ax.set_yticks(y); ax.set_yticklabels(eds, fontsize=8); ax.invert_yaxis()
ax.set_xlim(-1.05, 1.05); ax.set_xlabel("Edit Index  (−1 = unedited world · +1 = edited world)")
ax.set_title("(a) Edit Index — bounded, differing-pixel support,\nso this panel IS cross-channel comparable", fontsize=9.5)
# lower LEFT: the oracle bars run far to the right at the bottom of this panel,
# so a right-hand legend sits on top of them.
ax.legend(fontsize=7, frameon=False, loc="lower left"); style_ax(ax)

ax = axes[1]
for j, (name, M) in enumerate(MODELS.items()):
    base = CARDS[(name, "Unsteered")]["edit_index"]
    ax.barh(y + j*hgt - .4 + hgt/2, [CARDS[(name, e)]["edit_index"] - base for e in eds],
            height=hgt, color=M["color"])
ax.axvline(0, color="#444", lw=1)
ax.set_yticks(y); ax.set_yticklabels([]); ax.invert_yaxis()
ax.set_xlabel("Edit Index − that arm's own unsteered index  (index points)")
ax.set_title("(b) gain over doing nothing, each arm against\nits OWN −1 end", fontsize=9.5); style_ax(ax)

ax = axes[2]
# One seed per channel here: seed robustness is already shown in (a)/(b), and nine
# overlapping curves made the legend unreadable on top of the lines.
for name in [RUNS[0][0], RUNS[2][0]]:
    M = MODELS[name]
    for ed, ls in [("Pseudoinverse Injection", "-"), ("Counterfactual Overwriting", "--"),
                   ("Decoder Grad Steering k=15", ":")]:
        ax.plot(CARDS[(name, ed)]["edit_index_by_step"], ls, color=M["color"], lw=1.7,
                label=f"{'2D' if M['ch']=='2d' else '1D'} · {ed}")
ax.axhline(0, color="#444", lw=1)
ax.set_xlabel("rollout step (step 0 = edit frame)"); ax.set_ylabel("Edit Index")
ax.set_title("(c) does the edit HOLD? index at every step\n(one standard editor, two oracles; seed 0 only)",
             fontsize=9.5)
ax.legend(fontsize=7, frameon=False, ncol=2, loc="upper center",
          bbox_to_anchor=(0.5, -0.20))          # BELOW the axes, never over the curves
style_ax(ax)
fig.suptitle("Fig 4 — editing head-to-head: does full observability change what can be edited?",
             fontsize=12, y=1.03)
plt.tight_layout(); fig.subplots_adjust(bottom=0.28); plt.show()

### §4 — the generations themselves

A scorecard compresses a rollout to one number and routinely hides the difference between *the edit
landed* and *the output degraded* — the two look identical in an Edit Index that moved. `CLAUDE.md`
therefore makes an observation-space panel mandatory for any claim about an effect on the
generations. Figs 5–6 are the omniscient-2D form of that requirement; the full spec and its
justification are in **`WATERFALL_SPEC_2D.md`**, and both figures are built by the single
`frame_grid.py` helper.

**Fig 5** shows raw model output and is the one that catches degradation. **Fig 6** composites all 15
rollout steps, so nothing is hidden by Fig 5's time subsample.

In [ ]:
# [17] Fig 5 — omniscient-2D frame grid (the waterfall analogue) for the main 2D arm.
MAIN = RUNS[0][0]
ch = MODELS[MAIN]["ch"]; d = E[ch]
s = int(np.argsort(-ZONES[ch].teleport)[2])          # a large, clearly visible teleport
show = ["Unsteered", "Pseudoinverse Injection", "MLP Grad Steering",
        "Counterfactual Overwriting", "Decoder Grad Steering k=15"]
arms = [Arm("GT (sim)\nclean render", GTROLL[ch], is_gt=True)] + [
    Arm(e.replace(" ", "\n", 1), ROLLS[(MAIN, e)],
        metric=f"Edit Index {CARDS[(MAIN, e)]['edit_index']:+.2f} · fid {CARDS[(MAIN, e)]['fidelity_ratio']:.2f}",
        leads_by_one=(e == "First Obs. TF")) for e in show]
fig = frame_grid(arms, cfg=CFG[ch], sample=s, ctx_obs=d["e"].obs[:N_EDIT], clean_obs=d["e"].clean_obs[:N_EDIT],
                 edit_frame=d["ef"], target_xy=d["target_xy"][s], ghost_xy=d["ghost_xy"][s],
                 fig_num="Fig 5", title=f"Omniscient 2D — generations by editor ({MAIN}, edit sample {s})")
plt.show()

In [ ]:
# [18] Fig 6 — trail composite: ALL 15 rollout steps per arm (nothing hidden by Fig 5's subsample).
fig = frame_trails(arms, cfg=CFG[ch], sample=s, clean_obs=d["e"].clean_obs[:N_EDIT], edit_frame=d["ef"],
                   n_steps=K_ROLL, target_xy=d["target_xy"][s], ghost_xy=d["ghost_xy"][s],
                   fig_num="Fig 6", title=f"Omniscient 2D — whole-rollout trails by editor ({MAIN}, sample {s})",
                   n_cols=3)
plt.show()

In [ ]:
# [19] Anim 1 — the same arms as an animation (the optional third view of the 2D spec).
#      Addition, never a replacement: Figs 5-6 are what the claim ships with, because a GIF
#      cannot be read in a committed notebook diff or a paper. What it adds is MOTION —
#      whether the edited object travels smoothly or snaps back is obvious in 3 seconds and
#      hard to read off five stills.
from IPython.display import Image as IPyImage
gif = frame_animation(arms[:4], cfg=CFG[ch], sample=s, ctx_obs=d["e"].obs[:N_EDIT],
                      clean_obs=d["e"].clean_obs[:N_EDIT], edit_frame=d["ef"],
                      path="anim1_editors_2d.gif", anim_num="Anim 1",
                      title=f"Omniscient 2D — editors over the rollout ({MAIN}, sample {s})",
                      target_xy=d["target_xy"][s], ghost_xy=d["ghost_xy"][s], n_steps=K_ROLL)
print(f"wrote {gif} — ~3 fps with ~1 s holds on the last pre-edit frame and the edit frame")
display(IPyImage(filename=gif))

In [ ]:
# [20] Fig 7 — the same editors in the 1D channel, same sample (identical scene, Table 1).
ch1 = "1d"; d1 = E[ch1]; M1 = RUNS[2][0]
fig, axes = plt.subplots(len(show) + 1, 1, figsize=(9.5, 1.5 * (len(show) + 1)), sharex=True)
fig.patch.set_facecolor("#0a0a14")
series = [("GT (sim)", GTROLL[ch1], None)] + [
    (e, ROLLS[(M1, e)], CARDS[(M1, e)]["edit_index"]) for e in show]
for ax, (nm, arr, ei) in zip(axes, series):
    ax.imshow(arr[s], cmap="gray", vmin=0, vmax=1, aspect="auto", interpolation="nearest")
    ax.set_yticks([]); ax.tick_params(colors="#a3adc2", labelsize=7)
    ax.set_ylabel(nm + ("" if ei is None else f"\n{ei:+.2f}"), color="#a3adc2",
                  fontsize=7.5, rotation=0, ha="right", va="center")
    for sp in ax.spines.values(): sp.set_color("#2a2f45")
axes[-1].set_xlabel("ray index", color="#a3adc2", fontsize=8)
fig.suptitle(f"Fig 7 — the SAME scene in the 1D channel ({M1}, sample {s})\n"
             "rows = editors, each panel: time (vertical, 15 steps) x rays (horizontal)",
             color="w", fontsize=10, y=1.005)
plt.tight_layout(); plt.show()
print("Fig 7 is a conventional 1D waterfall for the matched control arm; Figs 5-6 are its 2D analogue.")

---
## §5 — Summary

The one section that interprets. Measured quantities first (cell [21]), then the reading, marked as
interpretation and still quantified.

> ### Current results (updated 2026-08-11)
>
> **0. The pipeline is validated by the 1D control.** `1D_H256_30k_s0` reproduces the published
> `controls/H256` numbers (90k sequences, `../METRICS_AND_EDITORS.md`) almost exactly on 30k:
> Pseudoinverse Injection **−0.63** vs its unsteered **−0.66** (published −0.66 / −0.68),
> Counterfactual Overwriting **+0.68** (+0.70), Decoder Grad k=1 **+0.96** (+0.97), k=15 **+0.81**
> (+0.83), Freeze-time @8 **+0.52** (+0.52), First Obs. TF **−0.11** (−0.08). So the 30 000-sequence
> restriction costs essentially nothing, and any 1D↔2D difference below is not a sample-size effect.
>
> **1. The central negative SURVIVES full observability.** Best *standard* (training-free) editor gain
> over its own unsteered row: **+0.11 / +0.14** on the two omniscient arms versus **+0.13** in 1D.
> Pseudoinverse Injection is inert in both channels (**+0.02 / +0.02** vs **+0.03**), and every
> standard editor stays far on the unedited side. Removing *both* forms of observational loss —
> projection and occlusion — does not make the latent grabbable.
>
> **2. But the omniscient latent is LESS readable and LESS editable, not more.** Position R²
> **0.634 / 0.686** (linear) and **0.752 / 0.762** (MLP) versus **0.797 / 0.877** in 1D. Fiber
> residual **0.881 / 0.836** versus **0.583** — markedly *less* canonical. And every oracle weakens:
> Counterfactual Overwriting **+0.38 / +0.24** vs **+0.68**, Decoder Grad k=1 **+0.62 / +0.62** vs
> **+0.96**, Freeze-time **+0.34 / +0.27** vs **+0.52**. Best oracle gain **+1.16 / +1.14** vs **+1.62**.
>
> **3. Geometry moves the other way.** PCA hull @90 % **74 / 70** dims vs **44**, while TwoNN
> intrinsic dim is **2.3 / 2.5** vs **3.2**. A wider linear hull around a lower-dimensional estimate.
>
> **4. Seed robustness.** The two omniscient seeds agree to ≤0.14 index points on every editor and to
> ≤0.05 R² on position, so §1–§4 differences of that size carry no weight.
>
> ### Interpretation (2026-08-11) — clearly marked as interpretation
>
> Strictly more information produced a *worse* world state on every axis this thread measures. The
> leading candidate mechanism is **occupancy dilution, acting through the training objective**, and it
> is measured rather than assumed: an object covers **25.5 %** of the 1D scan but **1.45 %** of the
> omniscient frame (Table 2). The loss is a plain per-pixel MSE, so ~98.5 % of the omniscient gradient
> signal is about *background*, and the pressure to encode object position precisely is roughly
> **18× weaker per unit of loss**. The model duly reaches a *lower absolute* next-step RMSE (0.0875 vs
> 0.1051) at a similar ratio to its own noise floor (0.62× vs 0.68×) — largely by predicting empty
> space well. Fig 5 shows the consequence directly: relative to object size the omniscient generations
> are **soft blobs** where the 1D model's are crisp. The same blur explains the *less negative*
> unsteered index (−0.54 / −0.52 vs −0.66): `d_unedited` is inflated by the model's own blur on exactly
> the pixels the index scores.
>
> If that account is right, "omniscient is worse" is an **objective-weighting artifact, not a fact
> about observability** — and it is directly testable. Two pre-registered predictions: (a) reweighting
> the per-pixel loss by object occupancy, or (b) enlarging the objects (or shrinking the world) so
> occupancy approaches the 1D value, should recover position R² and oracle strength toward the 1D
> figures **while leaving the standard-editor result unchanged** (that one is the claim of result 1,
> and it should be robust). If instead the standard editors improve too, result 1 is confounded by
> blur and must be re-run at matched occupancy before it means anything.
>
> **What result 1 does and does not establish.** It shows the negative is not caused by *projection or
> occlusion*. It does **not** yet show the negative is independent of observation *sharpness*, because
> the two arms differ in effective blur as well as in channel. Occupancy-matched training is the
> control that separates them, and it is the first follow-on.
>
> **Not parameter-matched, by necessity.** Hidden size — the object under study — is matched at 256,
> but the encoder/decoder scale with the observation, so the omniscient arms carry 1.97 M parameters
> against the 1D arm's 0.46 M. Matching both is impossible when the observation dimension changes 24×;
> matching the latent is the right choice for a question about the latent, but the asymmetry is real
> and runs *in favour* of the 2D arm, which still reads worse.
>
> **Untested here:** the `∫gg′ = 0` geometry argument has not been re-derived for a 2D disc. In 1D an
> object's image is a plateau with two edge rays; in 2D it is a disc with a boundary *ring*, so the
> edge-to-interior mass ratio differs and the predicted cosine differs with it. That derivation would
> say whether result 1 is *predicted* or *surprising*, and it is the sharpest open question in this
> notebook.

In [ ]:
# [21] §5 — consolidated headline numbers, every arm, in one demarcated table.
rows = ["| quantity | " + " | ".join(MODELS) + " |", "|---|" + "---|" * len(MODELS)]
def row(lbl, fn, fmt="{:.3f}"):
    rows.append(f"| {lbl} | " + " | ".join(fmt.format(fn(n)) for n in MODELS) + " |")
row("next-step RMSE vs clean *(within-channel only)*", lambda n: Q[n]["next_step_clean"], "{:.4f}")
row("… as a ratio to its own noise floor", lambda n: Q[n]["next_step_clean"]/Q[n]["noise_floor"], "{:.2f}×")
row("PCA hull @90% (dims)", lambda n: GEO[n]["pca90"], "{:d}")
row("intrinsic dim (TwoNN)", lambda n: GEO[n]["twonn"], "{:.1f}")
row("position R² — linear", lambda n: REC[n]["pos"]["linear_r2"])
row("position R² — MLP", lambda n: REC[n]["pos"]["mlp_r2"])
row("velocity R² — MLP (late-t)", lambda n: REC[n]["vel_late"]["mlp_r2"])
row("fiber residual (MLP g), frac of ‖h‖", lambda n: FIB[n]["mlp"])
row("**Edit Index — Unsteered**", lambda n: CARDS[(n,"Unsteered")]["edit_index"], "{:+.2f}")
for ed in STD_ED:
    row(f"Edit Index — {ed}", lambda n, e=ed: CARDS[(n,e)]["edit_index"], "{:+.2f}")
    row(f"… gain over own unsteered", lambda n, e=ed: CARDS[(n,e)]["edit_index"]-CARDS[(n,"Unsteered")]["edit_index"], "{:+.2f}")
for ed in ORC_ED:
    row(f"Edit Index — {ed} *(oracle)*", lambda n, e=ed: CARDS[(n,e)]["edit_index"], "{:+.2f}")
row("best STANDARD editor gain (index points)",
    lambda n: max(CARDS[(n,e)]["edit_index"] for e in STD_ED) - CARDS[(n,"Unsteered")]["edit_index"], "{:+.2f}")
row("best ORACLE editor gain (index points)",
    lambda n: max(CARDS[(n,e)]["edit_index"] for e in ORC_ED) - CARDS[(n,"Unsteered")]["edit_index"], "{:+.2f}")
display(Markdown("## Consolidated headline numbers\n\n" + "\n".join(rows)))
print("\nCross-channel reading is restricted to the Edit Index, the R²/residual rows, and "
      "ratio-to-own-reference rows. RMSE rows are within-channel only (dilution warning).")